# Crypto ML Pipeline

This notebook demonstrates the ML pipeline for crypto trading using the local backtesting system.

## Overview
1. Load historical OHLCV data
2. Compute technical indicators (18 indicators)
3. Quantize momentum to [-1, 1]
4. Generate labels for future returns
5. Train a Random Forest model
6. Evaluate feature importance
7. Backtest using the trained model

In [ ]:
# Install dependencies (if running in Colab)
import subprocess
import sys

def install_if_missing(package, import_name=None):
    try:
        __import__(import_name or package)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

# Install required packages
install_if_missing("pandas")
install_if_missing("numpy")
install_if_missing("ta")
install_if_missing("scikit-learn")
install_if_missing("matplotlib")

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt

# Simulate OHLCV data (replace with real data in production)
np.random.seed(42)
n_days = 365
dates = pd.date_range(end=datetime.now(), periods=n_days, freq="1D")
price = 50000 + np.cumsum(np.random.randn(n_days) * 500)
price = np.maximum(price, 10000)  # Floor at 10k

df = pd.DataFrame({
    "open": price + np.random.randn(n_days) * 100,
    "high": price + abs(np.random.randn(n_days) * 200),
    "low": price - abs(np.random.randn(n_days) * 200),
    "close": price,
    "volume": np.random.exponential(1000, n_days),
}, index=dates)

print(f"Data shape: {df.shape}")
df.head()

In [ ]:
import sys
sys.path.insert(0, "/Users/unknown965/coding/OpenCodeTest/crypto-invest-analysis")

from feature_engine.indicators import compute_all_indicators
from feature_engine.momentum import momentum_score, momentum_delta
from feature_engine.labels import future_return, binary_label
from feature_engine.builder import build_feature_matrix

# Build feature matrix
features = build_feature_matrix(df, momentum_window=14, return_window=5)
print(f"Features shape: {features.shape}")
print(f"Columns: {list(features.columns)}")
features.head()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# Prepare data for ML
feature_cols = [col for col in features.columns if col not in ["close", "future_return", "binary_label"]]
X = features[feature_cols].dropna()
y = features.loc[X.index, "binary_label"].dropna()
X = X.loc[y.index]

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

In [ ]:
# Train model
model = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
# Feature importance
importance = pd.Series(model.feature_importances_, index=feature_cols)
importance = importance.sort_values(ascending=False)

plt.figure(figsize=(10, 8))
importance.head(20).plot(kind="barh")
plt.title("Top 20 Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

In [ ]:
# Backtest with model
from backtest_engine.model_strategy import ModelStrategy
from backtest_engine.engine import BacktestEngine
from backtest_engine.metrics import calculate_metrics, format_report

strategy = ModelStrategy(
    model=model,
    feature_columns=feature_cols,
    confidence_threshold=0.6,
)

engine = BacktestEngine(
    strategy=strategy,
    initial_capital=10000,
    fee_rate=0.001,
    slippage=0.0005,
    symbol="BTC/USDT",
    timeframe="1d",
)

result = engine.run(features)

print(f"Total trades: {result.total_trades}")
print(f"Final equity: ${result.final_equity:,.2f}")
print(f"Total return: {result.total_return_pct:.1f}%")
print(f"Max drawdown: {result.max_drawdown_pct:.1f}%")
print(f"Win rate: {result.win_rate:.1f}%")
print(f"Sharpe ratio: {result.sharpe_ratio:.2f}")

In [ ]:
# Plot equity curve
plt.figure(figsize=(12, 6))
plt.plot(result.equity_curve)
plt.title("Equity Curve")
plt.xlabel("Trading Days")
plt.ylabel("Equity ($)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrates:
1. **Feature Engineering**: Computing 18 technical indicators and quantizing momentum
2. **Model Training**: Training a Random Forest classifier on labeled data
3. **Feature Importance**: Identifying which indicators drive predictions
4. **Backtesting**: Running the trained model through the backtest engine

### Next Steps
- Replace simulated data with real historical data
- Experiment with different models (XGBoost, LightGBM)
- Optimize hyperparameters
- Add more features (volume analysis, market regime detection)